# 智能数据分析师（Agent Data Analyst）

## 项目简介

本项目基于 **Hello-Agents** 框架构建了一个「智能数据分析师」Agent。用户只需用自然语言提出数据分析需求，Agent 即可自主完成 **数据加载 → 数据质量诊断 → 统计分析 → 数据清洗 → 可视化图表生成 → 业务洞察总结** 的完整分析流程，让不会编程的业务人员也能一键获得专业的数据分析报告。

**核心特性：**
- 🗣️ 自然语言交互：零代码完成数据分析
- 🔧 工具调用（Tool Use）：Agent 自主规划并调用 4 个专业数据分析工具
- 📊 自动可视化：根据分析意图自动选择合适图表类型
- 💡 业务洞察：不止给数字，更给出可执行的业务建议

## 作者信息
- 姓名：孟凡超
- GitHub：BeiXiao-929
- 日期：2026.9.3

## 项目结构

```
DataAnalystAgent/
├── main.ipynb              # 本文件：项目主程序
├── README.md               # 项目说明文档
├── requirements.txt        # 依赖清单
├── data/
│   └── sample_sales.csv    # 示例数据（100条记录，含缺失值/重复行）
└── outputs/                # 图表与分析结果输出目录
```


## 第2部分：环境配置

安装依赖并导入必要的库。运行前请在项目根目录配置 `.env` 文件：

```bash
# .env 示例（以智谱为例，也可使用 OpenAI / DashScope 等）
ZHIPU_API_KEY=your_api_key_here
```


In [ ]:
# 安装依赖
%pip install -q hello-agents[all]

# 导入必要的库
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import BaseTool
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
from dotenv import load_dotenv

# 解决 matplotlib 中文乱码
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]
plt.rcParams["axes.unicode_minus"] = False

# 加载环境变量
load_dotenv()

DATA_PATH = "data/sample_sales.csv"
print("环境配置完成 ✅")

## 第3部分：工具定义

为本项目定义 4 个专业数据分析工具，全部继承 `hello_agents.tools.BaseTool`：

| 工具 | 名称 | 功能 |
|------|------|------|
| `CSVLoaderTool` | `csv_loader` | 加载数据集，返回规模、字段、样例、缺失值、重复行诊断 |
| `DataStatsTool` | `data_stats` | 数值列描述性统计 + 类别列分布统计 |
| `DataCleanTool` | `data_cleaner` | 缺失值填充、去重，保存清洗后数据并返回报告 |
| `DataVizTool` | `data_visualizer` | 生成柱状图/折线图/饼图/直方图/散点图并保存到 outputs/ |


In [ ]:
class CSVLoaderTool(BaseTool):
    """数据加载工具：读取CSV并返回数据全貌与质量诊断"""

    name = "csv_loader"
    description = (
        "加载CSV数据集并返回数据概览，包括：数据规模、字段及类型、前5行样例、"
        "各列缺失值数量、重复行数量。"
        "输入为CSV文件路径，例如：data/sample_sales.csv"
    )

    def run(self, query: str) -> str:
        path = query.strip().strip("'\"")
        if not os.path.exists(path):
            return f"错误：文件 {path} 不存在，请检查路径"
        try:
            df = pd.read_csv(path)
        except Exception as e:
            return f"错误：读取文件失败 - {e}"

        missing = df.isnull().sum()
        missing = missing[missing > 0]
        info = {
            "文件路径": path,
            "数据规模": f"{df.shape[0]} 行 x {df.shape[1]} 列",
            "字段及类型": {col: str(dtype) for col, dtype in df.dtypes.items()},
            "前5行样例": df.head().to_dict(orient="records"),
            "缺失值统计": {k: int(v) for k, v in missing.items()} if len(missing) else "无缺失值",
            "重复行数量": int(df.duplicated().sum()),
        }
        return json.dumps(info, ensure_ascii=False, indent=2, default=str)


class DataStatsTool(BaseTool):
    """统计分析工具：数值列描述性统计 + 类别列分布"""

    name = "data_stats"
    description = (
        "对数据集进行统计分析：自动输出所有数值列的计数/均值/标准差/最小值/四分位数/最大值，"
        "以及类别列的取值分布(Top5)。"
        "输入为CSV文件路径；如需分析某一列，用 :: 指定，例如：data/sample_sales.csv::region"
    )

    def run(self, query: str) -> str:
        parts = [p.strip() for p in query.split("::")]
        path = parts[0].strip("'\"")
        if not os.path.exists(path):
            return f"错误：文件 {path} 不存在"
        df = pd.read_csv(path)

        result = {"文件": path}

        # 指定了单列分析
        if len(parts) > 1 and parts[1] and parts[1] in df.columns:
            col, series = parts[1], df[parts[1]]
            if pd.api.types.is_numeric_dtype(series):
                result["列统计"] = series.describe().round(2).to_dict()
            else:
                vc = series.value_counts(dropna=False).head(10)
                result["取值分布"] = {str(k): int(v) for k, v in vc.items()}
            return json.dumps(result, ensure_ascii=False, indent=2, default=str)

        # 全表统计
        numeric_cols = df.select_dtypes(include="number").columns.tolist()
        if numeric_cols:
            result["数值列统计"] = df[numeric_cols].describe().round(2).to_dict()
        cat_cols = df.select_dtypes(exclude="number").columns.tolist()
        cat_dist = {}
        for col in cat_cols:
            vc = df[col].value_counts().head(5)
            if len(vc) <= 10:  # 低基数的类别列才输出分布
                cat_dist[col] = {str(k): int(v) for k, v in vc.items()}
        if cat_dist:
            result["类别列分布(Top5)"] = cat_dist
        return json.dumps(result, ensure_ascii=False, indent=2, default=str)


class DataCleanTool(BaseTool):
    """数据清洗工具：缺失值填充 + 去重，保存清洗结果并返回报告"""

    name = "data_cleaner"
    description = (
        "清洗数据集：数值列缺失值用中位数填充、类别列缺失值用众数填充、删除重复行。"
        "清洗后的数据保存到 outputs/ 目录。输入为CSV文件路径，返回清洗报告。"
    )

    def run(self, query: str) -> str:
        path = query.strip().strip("'\"").split("::")[0]
        if not os.path.exists(path):
            return f"错误：文件 {path} 不存在"
        df = pd.read_csv(path)
        report = {"原始规模": f"{df.shape[0]} 行 x {df.shape[1]} 列"}

        # 1. 缺失值处理
        missing_before = df.isnull().sum()
        missing_before = missing_before[missing_before > 0]
        fill_detail = {}
        for col in missing_before.index:
            if pd.api.types.is_numeric_dtype(df[col]):
                fill_value = df[col].median()
                strategy = "中位数"
            else:
                fill_value = df[col].mode().iloc[0] if not df[col].mode().empty else "未知"
                strategy = "众数"
            df[col] = df[col].fillna(fill_value)
            fill_detail[col] = f"{int(missing_before[col])} 个缺失值已用{strategy}({fill_value})填充"
        report["缺失值处理"] = fill_detail if fill_detail else "无缺失值"

        # 2. 去重
        dup_count = int(df.duplicated().sum())
        if dup_count:
            df = df.drop_duplicates().reset_index(drop=True)
        report["重复行处理"] = f"删除 {dup_count} 条重复行"

        # 3. 保存
        os.makedirs("outputs", exist_ok=True)
        cleaned_path = f"outputs/cleaned_{os.path.basename(path)}"
        df.to_csv(cleaned_path, index=False, encoding="utf-8-sig")
        report["清洗后规模"] = f"{df.shape[0]} 行 x {df.shape[1]} 列"
        report["清洗后文件"] = cleaned_path
        return json.dumps(report, ensure_ascii=False, indent=2, default=str)


class DataVizTool(BaseTool):
    """可视化工具：根据分析意图生成图表并保存"""

    name = "data_visualizer"
    description = (
        "生成数据可视化图表。输入格式：文件路径|图表类型|x轴列|y轴列，"
        "图表类型支持：bar(柱状图,可省略y轴列则统计计数)、line(折线图)、"
        "pie(饼图,可省略y轴列)、hist(直方图,只需数值列)、scatter(散点图)。"
        "例如：data/sample_sales.csv|bar|region|sales_amount"
    )

    def run(self, query: str) -> str:
        parts = [p.strip() for p in query.split("|")]
        if len(parts) < 2:
            return "错误：输入格式应为 文件路径|图表类型|x轴列|y轴列"
        path = parts[0].strip("'\"")
        if not os.path.exists(path):
            return f"错误：文件 {path} 不存在"
        df = pd.read_csv(path)
        chart_type = parts[1].lower()
        x_col = parts[2] if len(parts) > 2 and parts[2] else None
        y_col = parts[3] if len(parts) > 3 and parts[3] else None

        # 数值化清洗：非数值的数值列（如空字符串）转为 NaN 后丢弃
        if y_col and y_col in df.columns:
            df[y_col] = pd.to_numeric(df[y_col], errors="coerce")
        df = df.dropna(subset=[c for c in [x_col, y_col] if c and c in df.columns])

        plt.figure(figsize=(10, 6))
        try:
            if chart_type == "bar":
                if y_col:
                    data = df.groupby(x_col)[y_col].sum().sort_values(ascending=False)
                    plt.bar(data.index.astype(str), data.values, color="#3498db")
                    plt.ylabel(y_col)
                else:
                    data = df[x_col].value_counts()
                    plt.bar(data.index.astype(str), data.values, color="#3498db")
                    plt.ylabel("数量")
                plt.title(f"{x_col} 分组统计（柱状图）")
            elif chart_type == "line":
                if x_col and df[x_col].apply(lambda v: str(v)[:1].isdigit()).mean() > 0.5:
                    df = df.sort_values(x_col)
                plt.plot(df[x_col].astype(str), df[y_col], marker="o", color="#e74c3c")
                plt.title(f"{y_col} 随 {x_col} 变化趋势（折线图）")
                plt.ylabel(y_col)
            elif chart_type == "pie":
                data = df.groupby(x_col)[y_col].sum() if y_col else df[x_col].value_counts()
                plt.pie(data.values, labels=data.index.astype(str), autopct="%1.1f%%")
                plt.title(f"{x_col} 占比（饼图）")
            elif chart_type == "hist":
                col = x_col if x_col in df.columns else y_col
                plt.hist(pd.to_numeric(df[col], errors="coerce").dropna(), bins=15, color="#2ecc71", edgecolor="white")
                plt.title(f"{col} 分布（直方图）")
                plt.xlabel(col)
            elif chart_type == "scatter":
                plt.scatter(df[x_col], df[y_col], alpha=0.6, color="#9b59b6")
                plt.xlabel(x_col)
                plt.ylabel(y_col)
                plt.title(f"{x_col} 与 {y_col} 的关系（散点图）")
            else:
                plt.close()
                return f"错误：不支持的图表类型 {chart_type}，支持 bar/line/pie/hist/scatter"
        except Exception as e:
            plt.close()
            return f"错误：绘图失败 - {e}，请检查列名是否正确"

        os.makedirs("outputs", exist_ok=True)
        filename = f"outputs/{chart_type}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.png"
        plt.xticks(rotation=30)
        plt.tight_layout()
        plt.savefig(filename, dpi=110, bbox_inches="tight")
        plt.close()
        return f"图表已生成并保存至：{filename}（{chart_type} 图，x={x_col}, y={y_col}），可直接向用户展示该图表路径"


# 注册工具清单，方便后续统一管理
TOOLS = [CSVLoaderTool, DataStatsTool, DataCleanTool, DataVizTool]
print(f"工具定义完成 ✅ 共 {len(TOOLS)} 个工具：")
for t in TOOLS:
    print(f"  - {t.name}: {t.description[:30]}...")

## 第4部分：智能体构建

创建 LLM 与 `SimpleAgent`，并通过精心设计的系统提示词赋予 Agent「数据分析师」的角色、工作流程和输出规范。

In [ ]:
# 创建LLM
llm = HelloAgentsLLM()

# 系统提示词：定义智能数据分析师的角色、流程与输出规范
SYSTEM_PROMPT = """你是一位拥有10年经验的资深智能数据分析师，专注于电商/零售销售数据分析。

## 你的工作流程（ReCo范式：推理-行动-观察循环）
1. **了解数据**：任何分析开始前，先调用 csv_loader 了解数据全貌与质量
2. **统计分析**：调用 data_stats 获取关键指标的描述性统计
3. **数据清洗**：若发现缺失值或重复行，调用 data_cleaner 清洗，后续分析基于清洗后的数据
4. **可视化**：调用 data_visualizer 生成图表辅助说明
   - 对比不同类别 → bar 柱状图
   - 展示趋势 → line 折线图
   - 展示占比 → pie 饼图
   - 展示分布 → hist 直方图
5. **总结洞察**：基于以上所有观察结果，输出结构化分析结论

## 输出规范
最终回答必须包含以下部分（使用Markdown）：
### 📊 数据概况
### 🔍 关键发现（至少3条，附具体数字支撑）
### 📈 可视化图表（给出图表保存路径）
### 💡 业务建议（可执行、有优先级）
"""

# 创建智能体
agent = SimpleAgent(
    name="智能数据分析师",
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
)

# 添加工具
agent.add_tool(CSVLoaderTool())
agent.add_tool(DataStatsTool())
agent.add_tool(DataCleanTool())
agent.add_tool(DataVizTool())

print(f"智能体「{agent.name}」构建完成 ✅ 已挂载 4 个数据分析工具")

## 第5部分：功能演示

### 示例1：基础功能 —— 数据概况速览

In [ ]:
print("=== 示例1：基础功能（数据概况速览） ===")
result = agent.run(
    "请帮我全面了解 data/sample_sales.csv 这个数据集："
    "数据规模多大？包含哪些字段？数据质量有没有问题？"
)
print(result)

### 示例2：复杂场景 —— 完整分析流水线

要求 Agent 自主完成「清洗 → 区域销售对比分析 → 可视化 → 业务洞察」的端到端分析。

In [ ]:
print("=== 示例2：复杂场景（端到端完整分析） ===")
result = agent.run(
    "请对 data/sample_sales.csv 做一次完整的销售分析："
    "先检查并清洗数据质量问题，然后对比各区域的销售表现，"
    "生成一张区域销售额柱状图，再分析客户满意度情况，"
    "最后总结关键发现并给出下半年的业务建议。"
)
print(result)

# 查看输出的图表
import glob
from IPython.display import Image, display
charts = sorted(glob.glob("outputs/bar_*.png"))
if charts:
    display(Image(filename=charts[-1]))

## 第6部分：性能评估

从两个维度评估项目质量：

1. **工具层正确性测试**：验证 4 个工具在正常/异常输入下均能正确返回
2. **Agent 层定性评估**：从准确性、完整性、格式规范性等维度人工打分

In [ ]:
# ---------- 1. 工具层自动化测试 ----------
def run_tool(tool_cls, query):
    return tool_cls().run(query)

test_cases = [
    # (用例名, 工具, 输入, 预期包含的关键字)
    ("加载-正常路径", CSVLoaderTool, DATA_PATH, "数据规模"),
    ("加载-路径不存在", CSVLoaderTool, "data/not_exist.csv", "错误"),
    ("统计-全表", DataStatsTool, DATA_PATH, "数值列统计"),
    ("统计-单列", DataStatsTool, f"{DATA_PATH}::region", "取值分布"),
    ("清洗-含缺失与重复", DataCleanTool, DATA_PATH, "重复行处理"),
    ("绘图-柱状图", DataVizTool, f"{DATA_PATH}|bar|region|sales_amount", "图表已生成"),
    ("绘图-不支持类型", DataVizTool, f"{DATA_PATH}|radar|region|sales_amount", "错误"),
]

passed = 0
print(f"{'用例':<18}{'结果':<6}说明")
print("-" * 60)
for name, tool_cls, query, expected in test_cases:
    try:
        output = run_tool(tool_cls, query)
        ok = expected in output
    except Exception as e:
        ok, output = False, str(e)
    passed += ok
    print(f"{name:<18}{'✅' if ok else '❌':<6}{output.splitlines()[0][:40]}")

print(f"\n工具层测试：{passed}/{len(test_cases)} 通过")

# ---------- 2. Agent 层定性评估维度（人工打分表） ----------
print("""
Agent 层评估维度（满分5分，人工评估）：
| 维度           | 说明                                   | 得分 |
|----------------|----------------------------------------|------|
| 工具选择准确性 | 是否选用了正确的工具与参数              |  _   |
| 分析完整性     | 是否覆盖了用户要求的全部分析环节         |  _   |
| 洞察价值       | 结论是否有数字支撑、建议是否可执行       |  _   |
| 输出规范性     | 是否遵循系统提示词规定的Markdown结构     |  _   |
""")

## 第7部分：总结与展望

### 实现的功能
- ✅ 自然语言驱动的端到端数据分析：加载 → 诊断 → 清洗 → 统计 → 可视化 → 洞察
- ✅ 4 个专业工具（csv_loader / data_stats / data_cleaner / data_visualizer），覆盖数据分析全流程
- ✅ 智能图表选型：Agent 根据分析意图自动选择柱状/折线/饼图/直方图/散点图
- ✅ 数据质量自动诊断与修复：缺失值（中位数/众数填充）、重复行删除
- ✅ 结构化分析报告输出：数据概况 / 关键发现 / 图表 / 业务建议

### 遇到的挑战
- **挑战1：LLM 工具参数格式不稳定** —— LLM 偶尔会把路径和参数拼接成自然语言传给工具。
  *解决方案*：在工具 description 中给出精确的输入格式示例，并在 `run()` 内部做字符串清洗（去引号、容错解析）。
- **挑战2：matplotlib 中文乱码** —— 默认字体不含中文。
  *解决方案*：全局设置 `font.sans-serif = ["SimHei", "Microsoft YaHei"]`。
- **挑战3：脏数据导致绘图崩溃** —— 空字符串被读成 object 列，直接绘图报错。
  *解决方案*：在可视化工具内先 `pd.to_numeric(errors="coerce")` 再 `dropna`。

### 未来改进方向
- 支持多轮对话式分析（基于 `SessionAgent` 维护上下文，支持追问与下钻）
- 接入 SQL 数据源与 Excel 文件，扩展数据接入能力
- 增加异常检测、相关性分析、简单预测等高级分析工具
- 输出自动生成 PDF/HTML 格式的完整分析报告
- 引入多 Agent 协作：清洗 Agent、统计 Agent、报告 Agent 分工协作
